# Market Correlation Deep Dive

This notebook maps how markets move together in the training data, with a focus on:

- target correlation structure over time
- intraday and lag relationships between markets
- cross-market behavior of core weather/forecast features
- how feature values in one market relate to targets in another


In [ ]:
import itertools
import math
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.options.display.max_columns = 200

In [ ]:
DATA_PATH = Path('../data/train.csv')

df = pd.read_csv(DATA_PATH, parse_dates=['delivery_start', 'delivery_end'])
df = df.sort_values(['delivery_start', 'market']).reset_index(drop=True)

id_cols = {'id', 'target', 'market', 'delivery_start', 'delivery_end'}
feature_cols = [c for c in df.columns if c not in id_cols]
markets = sorted(df['market'].unique().tolist())

print(f'Rows: {len(df):,}')
print(f'Markets ({len(markets)}): {markets}')
print(f'Feature columns ({len(feature_cols)}):')
print(feature_cols)

dupe_pairs = df.duplicated(subset=['delivery_start', 'market']).sum()
print(f'Duplicate market-timestamp rows: {dupe_pairs:,}')

In [ ]:
coverage = (
    df.groupby('market')
    .agg(
        rows=('target', 'size'),
        start=('delivery_start', 'min'),
        end=('delivery_start', 'max'),
        target_mean=('target', 'mean'),
        target_std=('target', 'std'),
    )
    .sort_index()
)
coverage

In [ ]:
missing_share = (
    df[feature_cols + ['target']]
    .isna()
    .mean()
    .sort_values(ascending=False)
)
missing_share.head(15).to_frame('missing_fraction')

## Target Correlation Across Markets

Start by pivoting `target` into a wide panel (`timestamp x market`) and then compute static and dynamic correlation views.

In [ ]:
target_wide = (
    df.pivot_table(index='delivery_start', columns='market', values='target', aggfunc='mean')
    .reindex(columns=markets)
    .sort_index()
)

target_wide.head()

In [ ]:
pearson_corr = target_wide.corr(method='pearson')
spearman_corr = target_wide.corr(method='spearman')

for name, corr in [('Pearson', pearson_corr), ('Spearman', spearman_corr)]:
    fig = px.imshow(
        corr,
        text_auto='.2f',
        zmin=-1,
        zmax=1,
        color_continuous_scale='RdBu_r',
        title=f'Target Correlation Across Markets ({name})',
    )
    fig.update_layout(width=760, height=620)
    fig.show()

In [ ]:
ROLLING_WINDOW_HOURS = 24 * 14
pairs = list(itertools.combinations(markets, 2))

rolling_rows = []
for m1, m2 in pairs:
    s = target_wide[m1].rolling(ROLLING_WINDOW_HOURS, min_periods=ROLLING_WINDOW_HOURS // 2).corr(target_wide[m2])
    tmp = s.dropna().rename('corr').reset_index()
    tmp['pair'] = f'{m1} vs {m2}'
    rolling_rows.append(tmp)

rolling_corr_df = pd.concat(rolling_rows, ignore_index=True)

fig = px.line(
    rolling_corr_df,
    x='delivery_start',
    y='corr',
    color='pair',
    title=f'Rolling Target Correlation (window={ROLLING_WINDOW_HOURS} hours)',
)
fig.update_layout(width=1050, height=600, legend_title='Market Pair')
fig.show()

In [ ]:
monthly_rows = []
for period, chunk in target_wide.groupby(target_wide.index.to_period('M')):
    corr = chunk.corr()
    for m1, m2 in pairs:
        monthly_rows.append({
            'month': period.to_timestamp(),
            'pair': f'{m1} vs {m2}',
            'corr': corr.loc[m1, m2],
        })

monthly_corr_df = pd.DataFrame(monthly_rows)

fig = px.line(
    monthly_corr_df,
    x='month',
    y='corr',
    color='pair',
    markers=True,
    title='Monthly Target Correlation by Market Pair',
)
fig.update_layout(width=1050, height=600, legend_title='Market Pair')
fig.show()

In [ ]:
hourly_rows = []
for hour in range(24):
    chunk = target_wide[target_wide.index.hour == hour]
    corr = chunk.corr()
    for m1, m2 in pairs:
        hourly_rows.append({
            'hour': hour,
            'pair': f'{m1} vs {m2}',
            'corr': corr.loc[m1, m2],
        })

hourly_corr_df = pd.DataFrame(hourly_rows)

fig = px.line(
    hourly_corr_df,
    x='hour',
    y='corr',
    color='pair',
    markers=True,
    title='Target Correlation Regime by Hour of Day',
)
fig.update_layout(width=1050, height=600, xaxis=dict(dtick=1), legend_title='Market Pair')
fig.show()

In [ ]:
LAG_RANGE = range(-24, 25)
anchor_market = 'Market A' if 'Market A' in markets else markets[0]

lag_rows = []
for other in markets:
    if other == anchor_market:
        continue
    for lag in LAG_RANGE:
        corr = target_wide[anchor_market].corr(target_wide[other].shift(lag))
        lag_rows.append({
            'pair': f'{anchor_market} vs {other}',
            'lag_hours': lag,
            'corr': corr,
        })

lag_corr_df = pd.DataFrame(lag_rows)

fig = px.line(
    lag_corr_df,
    x='lag_hours',
    y='corr',
    color='pair',
    markers=True,
    title=f'Lead-Lag Target Correlation vs {anchor_market}',
)
fig.add_vline(x=0, line_dash='dash', line_color='black')
fig.update_layout(width=980, height=560, legend_title='Pair')
fig.show()

best_lag_idx = (
    lag_corr_df.assign(abs_corr=lag_corr_df['corr'].abs())
    .groupby('pair')['abs_corr']
    .idxmax()
)
best_lag = (
    lag_corr_df.loc[best_lag_idx]
    .assign(abs_corr=lambda x: x['corr'].abs())
    .sort_values('abs_corr', ascending=False)
    .reset_index(drop=True)
)
best_lag

## Feature Correlation Across Markets

For each numeric feature, we compute market-to-market correlations to identify which signals are shared globally vs locally noisy.

In [ ]:
def upper_triangle_values(corr_df: pd.DataFrame) -> pd.Series:
    mask = np.triu(np.ones(corr_df.shape, dtype=bool), k=1)
    return corr_df.where(mask).stack()

feature_wide_cache = {}
feature_strength_rows = []

for feature in feature_cols:
    wide = (
        df.pivot_table(index='delivery_start', columns='market', values=feature, aggfunc='mean')
        .reindex(columns=markets)
        .sort_index()
    )
    feature_wide_cache[feature] = wide

    corr = wide.corr()
    vals = upper_triangle_values(corr)
    feature_strength_rows.append({
        'feature': feature,
        'mean_corr': vals.mean(),
        'mean_abs_corr': vals.abs().mean(),
        'std_corr': vals.std(),
        'min_corr': vals.min(),
        'max_corr': vals.max(),
    })

feature_strength_df = pd.DataFrame(feature_strength_rows).sort_values('mean_abs_corr', ascending=False)
feature_strength_df.head(15)

In [ ]:
top_n = 10
top_features = feature_strength_df.head(top_n).iloc[::-1]
bottom_features = feature_strength_df.tail(top_n).sort_values('mean_abs_corr')

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('Most Shared Cross-Market Features', 'Least Shared Cross-Market Features'),
)

fig.add_trace(
    go.Bar(
        x=top_features['mean_abs_corr'],
        y=top_features['feature'],
        orientation='h',
        marker_color='#1f77b4',
        name='Top',
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Bar(
        x=bottom_features['mean_abs_corr'],
        y=bottom_features['feature'],
        orientation='h',
        marker_color='#d62728',
        name='Bottom',
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.update_layout(width=1200, height=520, title='Feature Cross-Market Correlation Strength (mean |corr|)')
fig.update_xaxes(title_text='mean |corr|', row=1, col=1)
fig.update_xaxes(title_text='mean |corr|', row=1, col=2)
fig.show()

In [ ]:
representative_features = feature_strength_df.head(3)['feature'].tolist() + feature_strength_df.tail(3)['feature'].tolist()

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=representative_features,
    horizontal_spacing=0.06,
    vertical_spacing=0.14,
)

for i, feature in enumerate(representative_features):
    row = i // 3 + 1
    col = i % 3 + 1
    corr = feature_wide_cache[feature].corr()

    fig.add_trace(
        go.Heatmap(
            z=corr.values,
            x=corr.columns,
            y=corr.index,
            zmin=-1,
            zmax=1,
            colorscale='RdBu_r',
            showscale=(i == 0),
            colorbar=dict(title='corr') if i == 0 else None,
        ),
        row=row,
        col=col,
    )

fig.update_layout(height=840, width=1220, title='Feature Correlation Matrices: High vs Low Cross-Market Coupling')
fig.show()

In [ ]:
target_panel = target_wide.copy()

preferred = [
    'solar_forecast',
    'wind_forecast',
    'load_forecast',
    'air_temperature_2m',
    'wind_speed_10m',
    'cloud_cover_total',
]
selected_features = [f for f in preferred if f in feature_cols]
if len(selected_features) < 4:
    selected_features = feature_strength_df.head(4)['feature'].tolist()

feature_target_corr_mats = {}
for feature in selected_features:
    wide = feature_wide_cache[feature]
    mat = pd.DataFrame(index=markets, columns=markets, dtype=float)
    for f_market in markets:
        for t_market in markets:
            mat.loc[f_market, t_market] = wide[f_market].corr(target_panel[t_market])
    feature_target_corr_mats[feature] = mat

n = len(selected_features)
cols = 2
rows = math.ceil(n / cols)

fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=[f'{f}: feature-market vs target-market corr' for f in selected_features],
    horizontal_spacing=0.08,
    vertical_spacing=0.12,
)

for i, feature in enumerate(selected_features):
    row = i // cols + 1
    col = i % cols + 1
    mat = feature_target_corr_mats[feature]
    fig.add_trace(
        go.Heatmap(
            z=mat.values,
            x=mat.columns,
            y=mat.index,
            zmin=-1,
            zmax=1,
            colorscale='RdBu_r',
            showscale=(i == 0),
            colorbar=dict(title='corr') if i == 0 else None,
        ),
        row=row,
        col=col,
    )

fig.update_layout(height=380 * rows + 120, width=1120, title='Cross-Market Feature-to-Target Correlation Maps')
fig.show()

summary_rows = []
for feature, mat in feature_target_corr_mats.items():
    arr = mat.to_numpy(dtype=float)
    diag = np.nanmean(np.diag(arr))
    off = np.nanmean(arr[~np.eye(len(markets), dtype=bool)])
    summary_rows.append({
        'feature': feature,
        'mean_same_market_corr': diag,
        'mean_cross_market_corr': off,
    })

pd.DataFrame(summary_rows).sort_values('mean_same_market_corr', ascending=False)

## Notes for Modeling

Use this notebook to decide where to share signal across markets:

- Strong, stable target pair correlations support global/shared components.
- Features with high cross-market coupling are good candidates for common transformations.
- Features with weak coupling may need market-specific handling.
- Lag structure can suggest lead/lag engineered features or cross-market exogenous inputs.

## Prediction Correlation (Latest Submission)

This section loads the most recently modified submission file and compares market prediction behavior in the same style as the target analysis above.

In [ ]:
from pathlib import Path

submission_candidates = [
    p for p in Path('../csv').glob('submission*.csv') if p.is_file()
]
if Path('../runs').exists():
    submission_candidates += [p for p in Path('../runs').glob('*/submission.csv') if p.is_file()]

if not submission_candidates:
    raise FileNotFoundError('No submission CSV files found in repo root or runs/*/submission.csv')

latest_submission_path = max(submission_candidates, key=lambda p: p.stat().st_mtime)
print(f'Using latest submission: {latest_submission_path}')

submission_df = pd.read_csv(latest_submission_path)
if 'id' not in submission_df.columns or 'target' not in submission_df.columns:
    raise ValueError('Submission must contain columns: id, target')

test_meta = pd.read_csv('../data/test_for_participants.csv', usecols=['id', 'market', 'delivery_start'])
test_meta['delivery_start'] = pd.to_datetime(test_meta['delivery_start'])

pred_df = (
    submission_df.rename(columns={'target': 'prediction'})
    .merge(test_meta, on='id', how='left')
)

missing_meta = pred_df['market'].isna().sum()
if missing_meta:
    print(f'Warning: {missing_meta:,} submission rows missing market metadata from test set')

pred_df = pred_df.dropna(subset=['market', 'delivery_start'])

pred_wide = (
    pred_df.pivot_table(index='delivery_start', columns='market', values='prediction', aggfunc='mean')
    .reindex(columns=markets)
    .sort_index()
)

pred_markets = [m for m in pred_wide.columns if pred_wide[m].notna().any()]
pred_pairs = list(itertools.combinations(pred_markets, 2))

print(f'Prediction rows after merge: {len(pred_df):,}')
print(f'Prediction timestamps: {pred_wide.index.min()} -> {pred_wide.index.max()}')
print(f'Markets in prediction panel: {pred_markets}')

pred_wide.head()

In [ ]:
if len(pred_markets) < 2:
    print('Not enough markets in latest submission to compute cross-market correlation.')
else:
    pred_pearson = pred_wide[pred_markets].corr(method='pearson')
    pred_spearman = pred_wide[pred_markets].corr(method='spearman')

    for name, corr in [('Pearson', pred_pearson), ('Spearman', pred_spearman)]:
        fig = px.imshow(
            corr,
            text_auto='.2f',
            zmin=-1,
            zmax=1,
            color_continuous_scale='RdBu_r',
            title=f'Prediction Correlation Across Markets ({name})',
        )
        fig.update_layout(width=760, height=620)
        fig.show()

In [ ]:
if len(pred_pairs) == 0:
    print('No market pairs available for rolling/monthly/hourly prediction correlation.')
else:
    pred_len = len(pred_wide)
    pred_window = min(24 * 14, max(24, pred_len // 3))

    rolling_rows = []
    for m1, m2 in pred_pairs:
        s = pred_wide[m1].rolling(pred_window, min_periods=max(12, pred_window // 2)).corr(pred_wide[m2])
        tmp = s.dropna().rename('corr').reset_index()
        if tmp.empty:
            continue
        tmp['pair'] = f'{m1} vs {m2}'
        rolling_rows.append(tmp)

    if rolling_rows:
        pred_rolling_corr_df = pd.concat(rolling_rows, ignore_index=True)
        fig = px.line(
            pred_rolling_corr_df,
            x='delivery_start',
            y='corr',
            color='pair',
            title=f'Rolling Prediction Correlation (window={pred_window} hours)',
        )
        fig.update_layout(width=1050, height=600, legend_title='Market Pair')
        fig.show()
    else:
        print('Not enough history for rolling prediction correlation with the selected window.')

    monthly_rows = []
    for period, chunk in pred_wide[pred_markets].groupby(pred_wide.index.to_period('M')):
        corr = chunk.corr()
        for m1, m2 in pred_pairs:
            monthly_rows.append({'month': period.to_timestamp(), 'pair': f'{m1} vs {m2}', 'corr': corr.loc[m1, m2]})

    pred_monthly_corr_df = pd.DataFrame(monthly_rows)
    if not pred_monthly_corr_df.empty:
        fig = px.line(
            pred_monthly_corr_df,
            x='month',
            y='corr',
            color='pair',
            markers=True,
            title='Monthly Prediction Correlation by Market Pair',
        )
        fig.update_layout(width=1050, height=600, legend_title='Market Pair')
        fig.show()

    hourly_rows = []
    for hour in range(24):
        chunk = pred_wide[pred_markets][pred_wide.index.hour == hour]
        if chunk.shape[0] < 2:
            continue
        corr = chunk.corr()
        for m1, m2 in pred_pairs:
            hourly_rows.append({'hour': hour, 'pair': f'{m1} vs {m2}', 'corr': corr.loc[m1, m2]})

    pred_hourly_corr_df = pd.DataFrame(hourly_rows)
    if not pred_hourly_corr_df.empty:
        fig = px.line(
            pred_hourly_corr_df,
            x='hour',
            y='corr',
            color='pair',
            markers=True,
            title='Prediction Correlation Regime by Hour of Day',
        )
        fig.update_layout(width=1050, height=600, xaxis=dict(dtick=1), legend_title='Market Pair')
        fig.show()

In [ ]:
if len(pred_markets) < 2:
    print('Not enough markets for lead/lag prediction analysis.')
else:
    lag_range = range(-24, 25)
    anchor = 'Market A' if 'Market A' in pred_markets else pred_markets[0]

    lag_rows = []
    for other in pred_markets:
        if other == anchor:
            continue
        for lag in lag_range:
            corr = pred_wide[anchor].corr(pred_wide[other].shift(lag))
            lag_rows.append({'pair': f'{anchor} vs {other}', 'lag_hours': lag, 'corr': corr})

    pred_lag_corr_df = pd.DataFrame(lag_rows)
    if pred_lag_corr_df.empty:
        print('No lag correlations available.')
    else:
        fig = px.line(
            pred_lag_corr_df,
            x='lag_hours',
            y='corr',
            color='pair',
            markers=True,
            title=f'Lead-Lag Prediction Correlation vs {anchor}',
        )
        fig.add_vline(x=0, line_dash='dash', line_color='black')
        fig.update_layout(width=980, height=560, legend_title='Pair')
        fig.show()

        best_lag_idx = (
            pred_lag_corr_df.assign(abs_corr=pred_lag_corr_df['corr'].abs())
            .groupby('pair')['abs_corr']
            .idxmax()
        )
        pred_best_lag = (
            pred_lag_corr_df.loc[best_lag_idx]
            .assign(abs_corr=lambda x: x['corr'].abs())
            .sort_values('abs_corr', ascending=False)
            .reset_index(drop=True)
        )
        pred_best_lag